# Phase 6: Statistical Analysis of MCQ vs OSQ Evaluation Modalities

This notebook analyzes the performance differences between Multiple Choice Questions (MCQ) and Open Short Questions (OSQ) for evaluating LLM knowledge on systems engineering topics.

## Research Questions
1. **Position Bias**: Do MCQ answer positions (A/B/C/D) affect model performance?
2. **Modality Comparison**: Does evaluation modality (MCQ vs OSQ) significantly affect measured performance?
3. **Question Characteristics**: Do certain question types favor one modality over another?

## Dataset
- **Models**: gemma3__4b, gemma3__27b
- **MCQ Variants**: random, a, b, c, d (286 questions × 5 variants × 2 models)
- **OSQ**: Judged by openai_gpt-5 (limited samples, judging incomplete)
- **Correctness Threshold**: OSQ >= 70/100 points

## 1. Setup and Imports

In [ ]:
# Core data processing
import pandas as pd
import numpy as np
from pathlib import Path
import json

# Statistical analysis
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, ttest_rel, wilcoxon
from scipy.stats import mannwhitneyu, pearsonr, spearmanr
from sklearn.utils import resample

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Custom parsers
import sys
sys.path.append(str(Path.cwd()))
from parsers import parse_mcq_samples, parse_osq_judged_samples, align_mcq_osq_results

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✅ Imports complete")

## 2. Data Loading

In [ ]:
# Define paths
phase4_dir = Path("../phase4_inference/downloaded_output")
phase5_dir = Path("../phase5_llm_as_a_judge")

print("Loading data...")
print("=" * 80)

In [ ]:
# Parse MCQ data
print("\n📖 Parsing MCQ samples from Phase 4...")
mcq_data = parse_mcq_samples(phase4_dir, use_latest=True)
mcq_df = pd.DataFrame(mcq_data)

print(f"\n✅ MCQ DataFrame shape: {mcq_df.shape}")
print(f"   Models: {mcq_df['model'].unique().tolist()}")
print(f"   Variants: {mcq_df['variant'].unique().tolist()}")
print(f"   Questions: {mcq_df['question_id'].nunique()}")

In [ ]:
# Parse OSQ data
print("\n📖 Parsing OSQ judged samples from Phase 5...")
osq_data = parse_osq_judged_samples(phase5_dir, use_latest=True)
osq_df = pd.DataFrame(osq_data)

print(f"\n✅ OSQ DataFrame shape: {osq_df.shape}")
if len(osq_df) > 0:
    print(f"   Models: {osq_df['model'].unique().tolist()}")
    print(f"   Questions: {osq_df['question_id'].nunique()}")
    print(f"   Judge: {osq_df['judge_model'].unique().tolist()}")
else:
    print("   ⚠️  No OSQ data available (judging incomplete)")

In [ ]:
# Align MCQ and OSQ data
print("\n🔗 Aligning MCQ and OSQ results...")
aligned_data = align_mcq_osq_results(mcq_data, osq_data, missing_data_strategy='drop')
aligned_df = pd.DataFrame(aligned_data)

print(f"\n✅ Aligned DataFrame shape: {aligned_df.shape}")
print(f"   Records with MCQ data: {aligned_df['mcq_avg_correct'].notna().sum()}")
print(f"   Records with OSQ data: {aligned_df['osq_total_score'].notna().sum()}")
print(f"   Records with both: {(aligned_df['mcq_avg_correct'].notna() & aligned_df['osq_total_score'].notna()).sum()}")

## 3. Data Exploration

In [ ]:
# MCQ DataFrame preview
print("MCQ DataFrame Sample:")
display(mcq_df.head())

print("\nMCQ DataFrame Info:")
print(mcq_df.info())

In [ ]:
# OSQ DataFrame preview (if available)
if len(osq_df) > 0:
    print("OSQ DataFrame Sample:")
    display(osq_df.head())
    
    print("\nOSQ DataFrame Info:")
    print(osq_df.info())
    
    # OSQ score distribution
    print("\nOSQ Score Distribution:")
    print(osq_df[['total_score', 'percentage', 'is_correct']].describe())
else:
    print("⚠️  OSQ data not available for exploration")

In [ ]:
# Aligned DataFrame preview
print("Aligned DataFrame Sample:")
display(aligned_df.head())

print("\nAligned DataFrame Summary:")
display(aligned_df.describe())

## 4. Position Bias Analysis

Analyze whether MCQ answer position (A/B/C/D) affects model performance.

In [ ]:
# Filter for fixed-position variants only (exclude random)
mcq_fixed = mcq_df[mcq_df['variant'].isin(['a', 'b', 'c', 'd'])].copy()

# Calculate accuracy by position and model
position_accuracy = mcq_fixed.groupby(['model', 'variant'])['is_correct'].agg(['mean', 'count', 'sum']).reset_index()
position_accuracy.columns = ['model', 'variant', 'accuracy', 'n_questions', 'n_correct']

print("Position Bias - Accuracy by Variant:")
print("=" * 80)
display(position_accuracy.pivot(index='model', columns='variant', values='accuracy'))

# Overall position statistics
overall_position = mcq_fixed.groupby('variant')['is_correct'].agg(['mean', 'std', 'count'])
print("\nOverall Position Statistics (across models):")
display(overall_position)

In [ ]:
# Chi-square test for position bias
print("\n📊 Chi-Square Test for Position Bias")
print("=" * 80)
print("H0: Answer position does not affect correctness\n")

for model in mcq_fixed['model'].unique():
    model_data = mcq_fixed[mcq_fixed['model'] == model]
    
    # Create contingency table: position × correct/incorrect
    contingency = pd.crosstab(model_data['variant'], model_data['is_correct'])
    
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    
    print(f"Model: {model}")
    print(f"  Chi-square: {chi2:.4f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  Result: {'Significant' if p_value < 0.05 else 'Not significant'} at α=0.05")
    print()

In [ ]:
# ANOVA test for position bias
print("\n📊 One-Way ANOVA for Position Bias")
print("=" * 80)
print("H0: Mean accuracy is equal across all positions\n")

for model in mcq_fixed['model'].unique():
    model_data = mcq_fixed[mcq_fixed['model'] == model]
    
    # Group by variant
    groups = [model_data[model_data['variant'] == v]['is_correct'].astype(int).values 
              for v in ['a', 'b', 'c', 'd']]
    
    # Perform one-way ANOVA
    f_stat, p_value = f_oneway(*groups)
    
    print(f"Model: {model}")
    print(f"  F-statistic: {f_stat:.4f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  Result: {'Significant' if p_value < 0.05 else 'Not significant'} at α=0.05")
    print()

In [ ]:
# Visualization: Position bias heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, model in enumerate(mcq_fixed['model'].unique()):
    model_data = mcq_fixed[mcq_fixed['model'] == model]
    
    # Calculate accuracy by position
    position_acc = model_data.groupby('variant')['is_correct'].mean().values.reshape(1, -1)
    
    # Create heatmap
    sns.heatmap(position_acc, annot=True, fmt='.3f', cmap='RdYlGn', 
                vmin=0, vmax=1, cbar_kws={'label': 'Accuracy'},
                xticklabels=['A', 'B', 'C', 'D'], yticklabels=[model],
                ax=axes[idx])
    axes[idx].set_title(f'Position Accuracy - {model}')
    axes[idx].set_xlabel('Answer Position')

plt.tight_layout()
plt.savefig('position_bias_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: position_bias_heatmap.png")

In [ ]:
# Visualization: Position bias bar chart
fig, ax = plt.subplots(figsize=(10, 6))

position_accuracy_wide = position_accuracy.pivot(index='variant', columns='model', values='accuracy')
position_accuracy_wide.plot(kind='bar', ax=ax, width=0.7)

ax.set_xlabel('Answer Position', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('MCQ Accuracy by Answer Position', fontsize=14, fontweight='bold')
ax.set_xticklabels(['A', 'B', 'C', 'D'], rotation=0)
ax.legend(title='Model', loc='best')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('position_bias_bar.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: position_bias_bar.png")

In [ ]:
# Visualization: Model accuracy with variance bars across positions
fig, ax = plt.subplots(figsize=(10, 6))

# Calculate mean accuracy and std across all positions for each model
model_stats = mcq_fixed.groupby('model')['is_correct'].agg(['mean', 'std', 'count']).reset_index()
model_stats.columns = ['model', 'mean_accuracy', 'std_accuracy', 'n_samples']

# Create bar plot
x_pos = np.arange(len(model_stats))
bars = ax.bar(x_pos, model_stats['mean_accuracy'], 
               yerr=model_stats['std_accuracy'],
               capsize=10, 
               alpha=0.8, 
               color='#2ecc71',
               edgecolor='black',
               linewidth=1.5,
               error_kw={'linewidth': 2, 'ecolor': 'black'})

# Customize plot
ax.set_xlabel('Model', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=13, fontweight='bold')
ax.set_title('Model Accuracy Across Answer Positions\n(Error bars show standard deviation)', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(model_stats['model'], rotation=0, fontsize=11)
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on top of bars
for i, (bar, mean_val, std_val) in enumerate(zip(bars, model_stats['mean_accuracy'], model_stats['std_accuracy'])):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + std_val + 0.02,
           f'{mean_val:.3f}±{std_val:.3f}',
           ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('model_accuracy_with_variance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: model_accuracy_with_variance.png")
print("\nModel Statistics:")
display(model_stats)

## 5. MCQ vs OSQ Comparison

Compare performance between MCQ (average across positions) and OSQ evaluations.

In [ ]:
# Filter for records with both MCQ and OSQ data
both_modalities = aligned_df[
    aligned_df['mcq_avg_correct'].notna() & 
    aligned_df['osq_total_score'].notna()
].copy()

print(f"Records with both MCQ and OSQ data: {len(both_modalities)}")

if len(both_modalities) == 0:
    print("\n⚠️  Cannot perform MCQ vs OSQ comparison: No overlapping data")
    print("    OSQ judging is incomplete. Analysis will focus on MCQ position bias.")
else:
    print(f"\nModels in comparison: {both_modalities['model'].unique().tolist()}")
    print(f"Questions in comparison: {both_modalities['question_id'].nunique()}")
    
    # Convert OSQ score to binary (>= 70 = correct)
    both_modalities['osq_correct'] = both_modalities['osq_is_correct'].astype(int)
    
    display(both_modalities[['model', 'question_id', 'mcq_avg_correct', 
                             'osq_total_score', 'osq_correct']].head(10))

In [ ]:
# Skip MCQ vs OSQ comparison if no data
if len(both_modalities) > 0:
    # Summary statistics
    print("\n📊 MCQ vs OSQ Performance Summary")
    print("=" * 80)
    
    summary = both_modalities.groupby('model').agg({
        'mcq_avg_correct': ['mean', 'std', 'count'],
        'osq_total_score': ['mean', 'std', 'count'],
        'osq_correct': ['mean', 'std']
    }).round(4)
    
    display(summary)
    
    # Overall statistics
    print("\nOverall (across models):")
    print(f"  MCQ Average Accuracy: {both_modalities['mcq_avg_correct'].mean():.4f} ± {both_modalities['mcq_avg_correct'].std():.4f}")
    print(f"  OSQ Average Score: {both_modalities['osq_total_score'].mean():.2f} ± {both_modalities['osq_total_score'].std():.2f}")
    print(f"  OSQ Binary Accuracy (>=70): {both_modalities['osq_correct'].mean():.4f} ± {both_modalities['osq_correct'].std():.4f}")

In [ ]:
# Paired t-test: MCQ vs OSQ
if len(both_modalities) > 0:
    print("\n📊 Paired T-Test: MCQ vs OSQ")
    print("=" * 80)
    print("H0: No difference in performance between MCQ and OSQ\n")
    
    # Test MCQ vs OSQ binary correctness
    t_stat, p_value = ttest_rel(both_modalities['mcq_avg_correct'], 
                                 both_modalities['osq_correct'])
    
    mean_diff = (both_modalities['mcq_avg_correct'] - both_modalities['osq_correct']).mean()
    
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  Mean difference (MCQ - OSQ): {mean_diff:.4f}")
    print(f"  Result: {'Significant' if p_value < 0.05 else 'Not significant'} at α=0.05")
    
    # Cohen's d (effect size)
    diff = both_modalities['mcq_avg_correct'] - both_modalities['osq_correct']
    cohens_d = diff.mean() / diff.std()
    print(f"\n  Cohen's d: {cohens_d:.4f}")
    
    if abs(cohens_d) < 0.2:
        effect = "negligible"
    elif abs(cohens_d) < 0.5:
        effect = "small"
    elif abs(cohens_d) < 0.8:
        effect = "medium"
    else:
        effect = "large"
    print(f"  Effect size: {effect}")

In [ ]:
# Wilcoxon signed-rank test (non-parametric alternative)
if len(both_modalities) > 0:
    print("\n📊 Wilcoxon Signed-Rank Test: MCQ vs OSQ")
    print("=" * 80)
    print("Non-parametric test for paired samples\n")
    
    w_stat, w_pvalue = wilcoxon(both_modalities['mcq_avg_correct'], 
                                 both_modalities['osq_correct'])
    
    print(f"  W-statistic: {w_stat:.4f}")
    print(f"  p-value: {w_pvalue:.4f}")
    print(f"  Result: {'Significant' if w_pvalue < 0.05 else 'Not significant'} at α=0.05")

In [ ]:
# Bootstrap confidence intervals
if len(both_modalities) > 0:
    print("\n📊 Bootstrap 95% Confidence Intervals")
    print("=" * 80)
    
    n_bootstrap = 10000
    np.random.seed(42)
    
    mcq_boots = []
    osq_boots = []
    diff_boots = []
    
    for _ in range(n_bootstrap):
        sample = resample(both_modalities, n_samples=len(both_modalities), random_state=None)
        mcq_boots.append(sample['mcq_avg_correct'].mean())
        osq_boots.append(sample['osq_correct'].mean())
        diff_boots.append(sample['mcq_avg_correct'].mean() - sample['osq_correct'].mean())
    
    mcq_ci = np.percentile(mcq_boots, [2.5, 97.5])
    osq_ci = np.percentile(osq_boots, [2.5, 97.5])
    diff_ci = np.percentile(diff_boots, [2.5, 97.5])
    
    print(f"\nMCQ Accuracy:")
    print(f"  Mean: {both_modalities['mcq_avg_correct'].mean():.4f}")
    print(f"  95% CI: [{mcq_ci[0]:.4f}, {mcq_ci[1]:.4f}]")
    
    print(f"\nOSQ Accuracy (binary):")
    print(f"  Mean: {both_modalities['osq_correct'].mean():.4f}")
    print(f"  95% CI: [{osq_ci[0]:.4f}, {osq_ci[1]:.4f}]")
    
    print(f"\nDifference (MCQ - OSQ):")
    print(f"  Mean: {mean_diff:.4f}")
    print(f"  95% CI: [{diff_ci[0]:.4f}, {diff_ci[1]:.4f}]")
    print(f"  CI includes 0: {diff_ci[0] <= 0 <= diff_ci[1]}")

In [ ]:
# Visualization: MCQ vs OSQ bar chart
if len(both_modalities) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Prepare data
    comparison_data = both_modalities.groupby('model').agg({
        'mcq_avg_correct': 'mean',
        'osq_correct': 'mean'
    }).reset_index()
    
    x = np.arange(len(comparison_data))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, comparison_data['mcq_avg_correct'], width, 
                   label='MCQ', color='#3498db', alpha=0.8)
    bars2 = ax.bar(x + width/2, comparison_data['osq_correct'], width, 
                   label='OSQ', color='#e74c3c', alpha=0.8)
    
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title('MCQ vs OSQ Performance Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_data['model'], rotation=45, ha='right')
    ax.legend(loc='best')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('mcq_vs_osq_bar.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved: mcq_vs_osq_bar.png")

In [ ]:
# Visualization: MCQ vs OSQ scatter plot
if len(both_modalities) > 0:
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Scatter plot by model
    for model in both_modalities['model'].unique():
        model_data = both_modalities[both_modalities['model'] == model]
        ax.scatter(model_data['mcq_avg_correct'], model_data['osq_correct'], 
                  label=model, alpha=0.6, s=100)
    
    # Add diagonal line (perfect agreement)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect agreement')
    
    # Calculate and plot regression line
    from scipy.stats import linregress
    slope, intercept, r_value, p_value, std_err = linregress(
        both_modalities['mcq_avg_correct'], 
        both_modalities['osq_correct']
    )
    x_line = np.array([0, 1])
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, 'r-', alpha=0.5, 
           label=f'Regression (r={r_value:.3f}, p={p_value:.3f})')
    
    ax.set_xlabel('MCQ Accuracy', fontsize=12)
    ax.set_ylabel('OSQ Accuracy (Binary)', fontsize=12)
    ax.set_title('MCQ vs OSQ Performance Correlation', fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig('mcq_vs_osq_scatter.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved: mcq_vs_osq_scatter.png")

## 6. Question-Level Analysis

In [ ]:
# Question-level consistency (if OSQ data available)
if len(both_modalities) > 0:
    print("\n📊 Question-Level Consistency")
    print("=" * 80)
    
    # Correlation between MCQ and OSQ difficulty
    question_difficulty = both_modalities.groupby('question_id').agg({
        'mcq_avg_correct': 'mean',
        'osq_correct': 'mean'
    }).reset_index()
    
    # Pearson correlation
    pearson_r, pearson_p = pearsonr(question_difficulty['mcq_avg_correct'], 
                                     question_difficulty['osq_correct'])
    
    # Spearman correlation (non-parametric)
    spearman_r, spearman_p = spearmanr(question_difficulty['mcq_avg_correct'], 
                                        question_difficulty['osq_correct'])
    
    print(f"\nPearson correlation: r = {pearson_r:.4f}, p = {pearson_p:.4f}")
    print(f"Spearman correlation: ρ = {spearman_r:.4f}, p = {spearman_p:.4f}")
    
    print("\nInterpretation: Do hard MCQ questions predict hard OSQ questions?")
    if abs(pearson_r) > 0.5 and pearson_p < 0.05:
        print("  ✅ Strong correlation - question difficulty is consistent across modalities")
    elif abs(pearson_r) > 0.3 and pearson_p < 0.05:
        print("  ⚠️  Moderate correlation - some consistency in question difficulty")
    else:
        print("  ❌ Weak/no correlation - question difficulty varies by modality")

## 7. Summary Report

In [ ]:
# Generate comprehensive summary
print("\n" + "=" * 80)
print(" " * 20 + "PHASE 6 ANALYSIS SUMMARY")
print("=" * 80)

print("\n📊 DATASET OVERVIEW")
print("-" * 80)
print(f"  MCQ Samples: {len(mcq_df):,}")
print(f"  OSQ Samples: {len(osq_df):,}")
print(f"  Aligned Records: {len(aligned_df):,}")
print(f"  Models: {', '.join(mcq_df['model'].unique())}")
print(f"  MCQ Variants: {', '.join(sorted(mcq_df['variant'].unique()))}")
print(f"  Unique Questions: {mcq_df['question_id'].nunique()}")

print("\n📊 POSITION BIAS ANALYSIS")
print("-" * 80)
print("  Position accuracy (overall):")
for variant in ['a', 'b', 'c', 'd']:
    acc = mcq_fixed[mcq_fixed['variant'] == variant]['is_correct'].mean()
    print(f"    Position {variant.upper()}: {acc:.4f}")

if len(both_modalities) > 0:
    print("\n📊 MCQ vs OSQ COMPARISON")
    print("-" * 80)
    print(f"  Records with both modalities: {len(both_modalities):,}")
    print(f"  MCQ Mean Accuracy: {both_modalities['mcq_avg_correct'].mean():.4f}")
    print(f"  OSQ Mean Accuracy: {both_modalities['osq_correct'].mean():.4f}")
    print(f"  Difference (MCQ - OSQ): {mean_diff:.4f}")
    print(f"  Statistical Significance: p = {p_value:.4f}")
    print(f"  Effect Size (Cohen's d): {cohens_d:.4f}")
else:
    print("\n⚠️  MCQ vs OSQ COMPARISON")
    print("-" * 80)
    print("  Not available - OSQ judging is incomplete")
    print("  Only 6 OSQ samples exist (3 per model)")
    print("  Need to complete judging for all 286 questions")

print("\n" + "=" * 80)

In [ ]:
# Export results
print("\n💾 Exporting Results...")
print("=" * 80)

# Create outputs directory
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# Save dataframes
mcq_df.to_csv(output_dir / "mcq_samples.csv", index=False)
print(f"✅ Saved: {output_dir}/mcq_samples.csv")

if len(osq_df) > 0:
    osq_df.to_csv(output_dir / "osq_samples.csv", index=False)
    print(f"✅ Saved: {output_dir}/osq_samples.csv")

aligned_df.to_csv(output_dir / "master_dataframe.csv", index=False)
print(f"✅ Saved: {output_dir}/master_dataframe.csv")

print("\n✅ Analysis complete!")

## 8. Additional Analyses (Optional)

Add additional cells below for:
- Category-specific performance breakdown
- Bloom's taxonomy level analysis (if OSQ data available)
- Model size effects (4b vs 27b)
- Custom visualizations